# Dental Vision V1 — Kaggle GPU diagnostic training
Runs DENTEX diagnostic training on Kaggle GPU. Selects diagnostic annotations only; never falls back to quadrant labels.


In [ ]:
!nvidia-smi


In [ ]:
import os
%cd /kaggle/working
if not os.path.exists('/kaggle/working/dental-vision-v1'):
    !git clone https://github.com/drhaidarali95/dental-vision-v1.git /kaggle/working/dental-vision-v1
%cd /kaggle/working/dental-vision-v1
!git pull
!pip -q install -r requirements.txt


In [ ]:
# Clean any partial/corrupt archives before downloading.
import pathlib
raw=pathlib.Path('data/dentex')
raw.mkdir(parents=True,exist_ok=True)
for p in raw.glob('*.zip'):
    print('Removing stale archive:',p)
    p.unlink()
!python scripts/download_dentex.py --out data/dentex


In [ ]:
import pathlib,zipfile
root=pathlib.Path('data/dentex')
for z in root.glob('*.zip'):
    dest=root/z.stem
    dest.mkdir(parents=True,exist_ok=True)
    print('Extracting',z,'->',dest)
    with zipfile.ZipFile(z) as f: f.extractall(dest)
!python scripts/inspect_dentex.py data/dentex


In [ ]:
import json,pathlib
wanted={'caries','deep caries','periapical lesion','periapical lesions','impacted tooth','impacted teeth'}
candidates=[]
for p in root.rglob('*.json'):
    try: d=json.loads(p.read_text())
    except Exception: continue
    if not isinstance(d,dict) or not {'images','annotations','categories'}.issubset(d): continue
    names={str(c.get('name','')).strip().lower() for c in d['categories']}
    score=len(names & wanted)
    print('COCO:',p,'images=',len(d['images']),'annotations=',len(d['annotations']),'categories=',sorted(names),'diagnostic_score=',score)
    if score: candidates.append((score,len(d['images']),p,d))
assert candidates,'No diagnostic COCO JSON found. Will not train quadrant labels.'
score,n,ann,d=max(candidates,key=lambda x:(x[0],x[1]))
assert score>=3,f'Only {score} expected diagnostic classes found'
print('SELECTED:',ann)
print('CATEGORIES:',[(c.get('id'),c.get('name')) for c in d['categories']])


In [ ]:
import subprocess,sys,pathlib
sample=d['images'][0]['file_name']
matches=list(root.rglob(pathlib.Path(sample).name))
assert matches,f'Could not locate sample image {sample}'
image_root=matches[0]
for _ in pathlib.Path(sample).parts: image_root=image_root.parent
out='/kaggle/working/dentex_diagnostic_v1.pt'
cmd=[sys.executable,'train.py','--images',str(image_root),'--annotations',str(ann),'--epochs','20','--batch-size','2','--output',out]
print('Launching:', ' '.join(cmd))
subprocess.run(cmd,check=True)
p=pathlib.Path(out)
assert p.exists(),'Checkpoint missing after training'
print('CHECKPOINT READY:',p,'MB=',round(p.stat().st_size/1024/1024,1))
print('Kaggle Output path:',p)
